Comentarios
↓
Limpieza texto
↓
Análisis de sentimiento
↓
Detección de tópicos
↓
Reglas de negocio
↓
Clasificación / alertas
↓
Modelo predictivo

In [1]:
!pip install pysentimiento
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pysentimiento import create_analyzer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_csv(
    '/content/drive/MyDrive/Proyecto/bbdd /fact_encuestas.csv'
)
# tomar muestra aleatoria de 100 comentarios
df_sample = df.sample(
    100,
    random_state=42
).copy()

Mounted at /content/drive


## Limpieza de texto

In [3]:
df['comentario'] = (
    df['comentario']
    .astype(str)
    .str.lower()
)

In [4]:
df['comentario'] = (
    df['comentario']
    .str.strip()
)

In [5]:
df = df[
    df['comentario'].str.len() > 0
]

## Crear analyzer

In [6]:
analyzer = create_analyzer(
    task="sentiment",
    lang="es"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

In [7]:
resultado = analyzer.predict(
    "La atención fue excelente"
)

print(resultado)

AnalyzerOutput(output=POS, probas={POS: 0.955, NEU: 0.040, NEG: 0.005})


In [8]:
resultado = analyzer.predict(
    "La demora fue terrible"
)

print(resultado)

AnalyzerOutput(output=NEG, probas={NEG: 0.924, NEU: 0.064, POS: 0.013})


In [9]:
# ============================================
# ANALISIS DE SENTIMIENTO
# ============================================

df_sample['sentimiento'] = (
    df_sample['comentario']
    .apply(
        lambda x:
        analyzer.predict(x).output
    )
)

## PASO 3 - Topics / embeddings

In [10]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.2 MB/s eta 0:00:00


In [11]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords



In [12]:
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
from nltk.corpus import stopwords

stopwords_es = stopwords.words('spanish')

print(stopwords_es[:20])

['de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo']


In [14]:
topic_model = BERTopic(
    language="multilingual"
)

In [15]:
topics, probs = topic_model.fit_transform(
    df_sample['comentario']
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
# ============================================
# AGREGAR TOPICS AL DATASET
# ============================================

df_sample['topic'] = topics

In [17]:
df.to_csv(
    '/content/drive/MyDrive/Proyecto/fact_encuestas_nlp.csv',
    index=False
)

In [18]:
# from google.colab import files

# files.download(
#     '/content/drive/MyDrive/Proyecto/fact_encuestas_nlp.csv'
# )

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# df_sample.to_excel(
#     '/content/drive/MyDrive/Proyecto/fact_encuestas_nlp.xlsx',
#     index=False
# )

In [20]:
# from google.colab import files

# files.download(
#     '/content/drive/MyDrive/Proyecto/fact_encuestas_nlp.xlsx'
# )

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1,-1_ecodopler_mamario_tardaron_mucho,"[ecodopler, mamario, tardaron, mucho, del, ate...",[Tardaron mucho en la atención del ecodopler m...
1,0,51,0_de_la_que_me,"[de, la, que, me, no, en, el, para, con, una]",[Desastre que no tengan en la guardia para hac...
2,1,48,1_excelente_muy_todo_atención,"[excelente, muy, todo, atención, la, bien, tod...",[todo muy bien excepto el concepto de humanida...
